# RT-LCOD — Run All local RTX A3000: dữ liệu thật → train → distill → test → video UI

Notebook này bám đúng kiến trúc đã chốt:

**YOLOE-26M (open-vocabulary) + lightweight Attribute/Relation Grounding Head + NO_TARGET + Grounding DINO teacher chỉ dùng offline.**

Mục tiêu của notebook là **bấm Run All một lần**. Nó tự chuẩn bị một tập gRefCOCO thực, cache proposal/feature, train Stage 1, chạy teacher distillation Stage 2, chọn checkpoint trên validation, test held-out, nạp runtime cuối và hiển thị UI video có ô nhập prompt.

Mọi session đều in log: môi trường/GPU, dữ liệu, proposal recall, loss/accuracy, checkpoint, thời gian, VRAM và latency. Giữ toàn bộ output của notebook để dùng cho vòng tối ưu tiếp theo.


## Session 0 — Cấu hình một chỗ duy nhất

Các giá trị mặc định được chọn cho RTX A3000 Laptop GPU 12 GB. Sau khi có log thật mới tăng dataset/epochs hoặc đổi input size.


In [9]:
# ===== USER CONTROLS — local RTX A3000 Laptop GPU 12 GB =====
RUN_PROFILE = "local_a3000"
TRAIN_SAMPLES = 1000
VAL_SAMPLES = 150
TEST_SAMPLES = 150
TEACHER_MAX_SAMPLES = 300     # Teacher offline; giữ nhỏ để ổn định trên 12 GB VRAM
DATA_DOWNLOAD_WORKERS = 4
PREFER_HTTP_COCO = True       # Mạng hiện tại lỗi TLS với CDN COCO; chỉ dùng HTTP công khai cho ảnh COCO
RUN_TEACHER = True            # False = bỏ Stage 2 để chạy nhanh hơn
REBUILD_CANDIDATE_CACHE = False

# test.mp4 ở thư mục gốc của project (ngoài rt_lcod).
VIDEO_PATH = "../test.mp4"
DEFAULT_PROMPT = "person"

print("[CONFIG] profile =", RUN_PROFILE)
print("[CONFIG] train/val/test =", TRAIN_SAMPLES, VAL_SAMPLES, TEST_SAMPLES)
print("[CONFIG] teacher_max_samples =", TEACHER_MAX_SAMPLES, "run_teacher =", RUN_TEACHER)
print("[CONFIG] data_download_workers =", DATA_DOWNLOAD_WORKERS)
print("[CONFIG] prefer_http_coco =", PREFER_HTTP_COCO)
print("[CONFIG] rebuild_candidate_cache =", REBUILD_CANDIDATE_CACHE)
print("[CONFIG] default video =", VIDEO_PATH)
print("[CONFIG] default prompt =", DEFAULT_PROMPT)


[CONFIG] profile = local_a3000
[CONFIG] train/val/test = 1000 150 150
[CONFIG] teacher_max_samples = 300 run_teacher = True
[CONFIG] data_download_workers = 4
[CONFIG] prefer_http_coco = True
[CONFIG] rebuild_candidate_cache = False
[CONFIG] default video = ../test.mp4
[CONFIG] default prompt = person


## Session 1 — Kiểm tra project local, runtime và tạo RunAll orchestrator

Cell này tự tìm project local và kiểm tra Python 3.14, CUDA cùng các thư viện đã cài sẵn. Notebook **không clone, không cập nhật Git và không tự cài thư viện**. Cuối cell tạo `runner`, là đối tượng điều phối toàn bộ pipeline.


In [10]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()

def find_root() -> Path | None:
    candidates = [HERE, HERE / "rt_lcod", HERE.parent, HERE.parent / "rt_lcod"]
    return next((path for path in candidates if (path / "pyproject.toml").is_file()), None)

ROOT = find_root()
if ROOT is None:
    raise RuntimeError("Không tìm thấy rt_lcod. Mở notebook từ thư mục training_test_model hoặc rt_lcod.")

try:
    import cv2, torch, yaml  # noqa: F401
except ImportError as exc:
    raise RuntimeError("Thiếu thư viện bắt buộc trong Python 3.14 (LCOD).") from exc

if not torch.cuda.is_available():
    raise RuntimeError("CUDA PyTorch không khả dụng. Hãy mở Jupyter bằng Python 3.14 (LCOD).")

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("[SETUP] ROOT =", ROOT)
print("[SETUP] Python =", sys.version.replace("\n", " "))
print("[SETUP] GPU =", torch.cuda.get_device_name(0))

from rt_lcod.runall import RunAll

runner = RunAll(
    ROOT,
    train_samples=TRAIN_SAMPLES,
    val_samples=VAL_SAMPLES,
    test_samples=TEST_SAMPLES,
    teacher_max_samples=TEACHER_MAX_SAMPLES,
    run_teacher=RUN_TEACHER,
    rebuild_candidate_cache=REBUILD_CANDIDATE_CACHE,
    config_name=RUN_PROFILE,
    data_download_workers=DATA_DOWNLOAD_WORKERS,
    prefer_http_coco=PREFER_HTTP_COCO,
)
print("[SETUP] RunAll ready")


[SETUP] ROOT = C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod
[SETUP] Python = 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
[SETUP] GPU = NVIDIA RTX A3000 12GB Laptop GPU
[SETUP] RunAll ready


## Session 2 — RTX A3000 / CUDA preflight

Dừng ngay nếu CUDA không khả dụng. Log này cho biết GPU, VRAM, CUDA/PyTorch và config inference/training thực tế.


In [11]:
runner.preflight()



[RUNALL] Session 1 — GPU/CUDA preflight
[PROGRESS] pipeline_session=1/11 completed=0%
[RUNALL] started=2026-09-29T08:25:51
[ENV] root=C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod
[ENV] python=3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
[ENV] platform=Windows-11-10.0.26200-SP0
[ENV] torch=2.11.0+cu128 cuda_runtime=12.8
[ENV] cuda_available=True
[ENV] gpu=NVIDIA RTX A3000 12GB Laptop GPU
NVIDIA RTX A3000 12GB Laptop GPU, 596.71, 12288 MiB, 11000 MiB, 12 %
[ENV] config=ExperimentConfig(seed=1337, run_name='rt_lcod_a3000_12gb', device='cuda', amp=True, model=ModelConfig(visual_dim=576, text_dim=384, hidden_dim=160, relation_feature_dim=11, dropout=0.1, top_k=16, region_encoder='mobilenet_v3_small', detector_name='yoloe-26m-seg.pt', text_model_name='sentence-transformers/all-MiniLM-L6-v2'), training=TrainingConfig(epochs=8, batch_size=32, learning_rate=0.0003, weight_decay=0.0001, grad_clip_norm=1.0, num_workers=2, save_every_epochs=1, early_st

## Session 3 — Dataset thật: gRefCOCO → canonical train/val/test

Tự tải metadata gRefCOCO và **chỉ các COCO image cần cho subset hiện tại**. V1 giữ single-target hoặc true no-target samples. Dataset được tách theo official split, sau đó validate schema và in thống kê/samples.


In [12]:
runner.prepare_data()



[RUNALL] Session 2 — download/prepare gRefCOCO
[PROGRESS] pipeline_session=2/11 completed=9%
[RUNALL] started=2026-09-29T08:25:51
[DATA][WARN] using public HTTP COCO download due to the configured local network TLS issue
[CMD] c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe scripts/prepare_grefcoco.py --root C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data --train 1000 --val 150 --test 150 --seed 1337 --negative-fraction 0.3 --workers 4 --allow-http-fallback --prefer-http-coco
[CMD] prepare_grefcoco: rc=0 elapsed_s=5.01
[CMD] c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe scripts/validate_manifest.py --manifest C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\manifests\train.jsonl
[CMD] validate_manifest: rc=0 elapsed_s=1.01
[DATA] split=train manifest=C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\manifests\train.jsonl bytes=451,702
[DATA] sample {"id": "gref_3875_315", "image": "images/train2014/COCO_train2014_000000

## Session 4 — YOLOE proposal + frozen visual/text cache

Mỗi sample chạy YOLOE-26M-seg một lần để lấy candidate. Sau đó MobileNetV3-Small + ROIAlign lấy region features và MiniLM lấy attribute/relation embedding. Cache giúp 10 epoch training không phải chạy lại YOLOE.

Quan trọng nhất ở đây là `positive_proposal_recall_at_iou`: nếu YOLOE không propose target thì grounding head không thể cứu.


In [13]:
runner.cache_candidates()



[RUNALL] Session 3 — YOLOE + frozen feature caches
[PROGRESS] pipeline_session=3/11 completed=18%
[RUNALL] started=2026-09-29T08:25:59
[CMD] c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe scripts/cache_candidates.py --config C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\configs\local_a3000.yaml --manifest C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\manifests\train.jsonl --output C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\candidate_cache\train --model yoloe-26m-seg.pt --confidence 0.18 --max-candidates 16
[PROGRESS] cache_candidates: still_running elapsed_m=0.5; use the script's tqdm/EPOCH line above for current percent and ETA.
[PROGRESS] cache_candidates: still_running elapsed_m=1.0; use the script's tqdm/EPOCH line above for current percent and ETA.
[PROGRESS] cache_candidates: still_running elapsed_m=1.5; use the script's tqdm/EPOCH line above for current percent and ETA.
[PROGRESS] cache_candidates: still_running elapsed_m=2

## Session 5 — Stage 1: train grounding student trên dữ liệu thật

YOLOE, MobileNet và MiniLM vẫn frozen. Chỉ train lightweight grounding head: visual projection, attribute score, relation score, candidate fusion và NO_TARGET. Trainer lưu `last.pt`, `best.pt`, checkpoint mỗi epoch, metrics và timing.

Sau train, validation chạy ngay và in metric.


In [14]:
runner.train_stage1()
stage1_val = runner.evaluate_stage1_validation()
print("[NOTEBOOK] Stage 1 validation:")
print(stage1_val)



[RUNALL] Session 4 — Stage 1 supervised student training
[PROGRESS] pipeline_session=4/11 completed=27%
[RUNALL] started=2026-09-29T08:42:12
[CMD] c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe scripts/train.py --config C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\configs\local_a3000.yaml --train-cache C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\candidate_cache\train --val-cache C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\candidate_cache\val --runs-root C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs --run-name stage1_local
[CMD] train: rc=0 elapsed_s=17.01
[STAGE1] run=C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs\20260929T014214Z_stage1_local
[STAGE1] best=C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs\20260929T014214Z_stage1_local\checkpoints\best.pt
[METRICS] file=C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs\20260929T014214Z_stage1_local\metrics.jsonl epochs=4
{
  "timestamp_

## Session 6 — Grounding DINO teacher cache (offline)

Teacher **không chạy khi deploy video**. Nó chỉ tạo soft distribution trên một subset train để student học thêm cross-modal grounding. Để phù hợp RTX A3000 12 GB, mặc định teacher dùng tối đa `TEACHER_MAX_SAMPLES`.


In [15]:
runner.cache_teacher_outputs()



[RUNALL] Session 6 — Grounding DINO offline teacher
[PROGRESS] pipeline_session=6/11 completed=45%
[RUNALL] started=2026-09-29T08:42:32
[CMD] c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe scripts/cache_teacher.py --manifest C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\manifests\train.jsonl --candidate-cache C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\candidate_cache\train --output C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\teacher_cache\train --model IDEA-Research/grounding-dino-base --threshold 0.2 --temperature 0.2 --max-samples 300
[CMD] cache_teacher: rc=0 elapsed_s=14.01
{
  "total": 300,
  "cached_new": 0,
  "cached_existing": 192,
  "missing_candidate": 108,
  "cache_files": 192,
  "mean_teacher_detections_new": 0.0,
  "elapsed_s": 10.765979700008756,
  "model": "IDEA-Research/grounding-dino-base"
}
[GPU][after-teacher] name=NVIDIA RTX A3000 12GB Laptop GPU total_GiB=12.00 alloc_MiB=0.0 reserved_MiB=0.0 peak_MiB=0.

## Session 7 — Stage 2: Knowledge Distillation

Khởi tạo student từ `best.pt` Stage 1 nhưng bắt đầu optimizer/scheduler mới. Các sample có teacher target dùng KL distillation; sample chưa có teacher vẫn dùng supervised losses bình thường.


In [16]:
runner.train_stage2()



[RUNALL] Session 7 — Stage 2 knowledge distillation
[PROGRESS] pipeline_session=7/11 completed=55%
[RUNALL] started=2026-09-29T08:42:47
[CMD] c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe scripts/train.py --config C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\configs\local_a3000.yaml --train-cache C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\candidate_cache\train --val-cache C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\candidate_cache\val --teacher-cache C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\teacher_cache\train --init-checkpoint C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs\20260929T014214Z_stage1_local\checkpoints\best.pt --runs-root C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs --run-name stage2_kd_local
[CMD] train: rc=0 elapsed_s=16.01
[STAGE2] run=C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs\20260929T014248Z_stage2_kd_local
[STAGE2] best=C:\Users\DELL\Desktop\VLM\tr

## Session 8 — Chọn model cuối bằng VALIDATION, không nhìn TEST

So Stage 1 và Stage 2 trên validation bằng composite `0.70 * target_top1 + 0.30 * no_target_accuracy`. Checkpoint tốt hơn được chọn làm final model.


In [17]:
FINAL_CHECKPOINT = runner.select_final_on_validation()
print("[NOTEBOOK] FINAL_CHECKPOINT =", FINAL_CHECKPOINT)



[RUNALL] Session 8 — validation model selection
[PROGRESS] pipeline_session=8/11 completed=64%
[RUNALL] started=2026-09-29T08:43:03
[CMD] c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe scripts/evaluate.py --config C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\configs\local_a3000.yaml --cache C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\candidate_cache\val --checkpoint C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs\20260929T014214Z_stage1_local\checkpoints\best.pt --output C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\artifacts\stage1_select_val.json
[CMD] evaluate: rc=0 elapsed_s=3.01
[EVAL] stage1_select/val={
  "target_top1_accuracy": 0.42391305643579236,
  "positive_target_accuracy": 0.4374063040899194,
  "no_target_accuracy": 0.0,
  "negative_false_positive_rate": 1.0,
  "samples": 92,
  "elapsed_s": 0.46558389998972416,
  "checkpoint": "C:\\Users\\DELL\\Desktop\\VLM\\training_test_model\\rt_lcod\\runs\\20260929T014214Z_s

## Session 9 — Held-out TEST một lần

Chỉ sau khi model đã được chọn trên validation mới đánh giá test. Log này là kết quả khách quan đầu tiên để so sánh các lần tối ưu sau.


In [18]:
final_test = runner.final_test()
print("[NOTEBOOK] FINAL TEST:")
print(final_test)



[RUNALL] Session 9 — held-out final test
[PROGRESS] pipeline_session=9/11 completed=73%
[RUNALL] started=2026-09-29T08:43:09
[CMD] c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe scripts/evaluate.py --config C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\configs\local_a3000.yaml --cache C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\data\candidate_cache\test --checkpoint C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs\20260929T014248Z_stage2_kd_local\checkpoints\best.pt --output C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\artifacts\final_test.json
[CMD] evaluate: rc=0 elapsed_s=5.02
[EVAL] final/test={
  "target_top1_accuracy": 0.3173076923076923,
  "positive_target_accuracy": 0.32767730492811936,
  "no_target_accuracy": 0.2222222238779068,
  "negative_false_positive_rate": 0.7777777910232544,
  "samples": 104,
  "elapsed_s": 2.202971000006073,
  "checkpoint": "C:\\Users\\DELL\\Desktop\\VLM\\training_test_model\\rt_lcod\\runs\\202609

## Session 10 — Load runtime cuối

Nạp checkpoint đã chọn + YOLOE-26M + frozen region/text encoders lên RTX A3000. Sau cell này model đã sẵn sàng cho video.


In [19]:
runtime = runner.load_runtime()



[RUNALL] Session 10 — load final deployment runtime
[PROGRESS] pipeline_session=10/11 completed=82%
[RUNALL] started=2026-09-29T08:43:14
[YOLOE] loaded model=yoloe-26m-seg.pt device=cuda imgsz=448 half=True


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[RUNTIME] device=cuda amp_fp16=True detector=yoloe-26m-seg.pt input=448 top_k=16
[RUNTIME] checkpoint=C:\Users\DELL\Desktop\VLM\training_test_model\rt_lcod\runs\20260929T014248Z_stage2_kd_local\checkpoints\best.pt
[RUNTIME] detector=yoloe-26m-seg.pt
[RUNTIME] input_size=448 top_k=16
[GPU][runtime-loaded] name=NVIDIA RTX A3000 12GB Laptop GPU total_GiB=12.00 alloc_MiB=226.8 reserved_MiB=248.0 peak_MiB=226.8
[RUNALL] load_runtime completed elapsed_s=5.60


## Session 11 — Video player + ô prompt

Đây là cell cuối của **Run All**. UI xuất hiện ngay cả khi `VIDEO_PATH` mặc định chưa tồn tại.

1. Nhập đường dẫn video vào ô **Video** (ví dụ `../test.mp4`).
2. Nhập prompt V1, ví dụ `red car`, `person next to bicycle`, `bottle left of laptop`.
3. Bấm **Play**.
4. Có thể đổi prompt trong lúc video đang chạy.
5. `Infer every = 1` là raw per-frame model; tăng lên 2–5 là chế độ thực dụng nhanh hơn.

Log bên dưới video ghi detector/region/head latency, candidate count, confidence, VRAM và cuối video có mean/p50/p95/p99/FPS.


In [20]:
player = runner.show_video_ui(VIDEO_PATH, DEFAULT_PROMPT)
print("[NOTEBOOK] Run All completed. The interactive video UI above is ready.")



[RUNALL] Session 11 — interactive video + prompt UI
[PROGRESS] pipeline_session=11/11 completed=91%
[RUNALL] started=2026-09-29T08:43:19


[VIDEO] UI ready. Type your video path and prompt, then click Play.
[VIDEO] infer every=1 is the raw per-frame model path; higher values are a practical speed mode.
[GPU][video-ui-ready] name=NVIDIA RTX A3000 12GB Laptop GPU total_GiB=12.00 alloc_MiB=226.8 reserved_MiB=248.0 peak_MiB=226.8
[RUNALL] show_video_ui completed elapsed_s=0.02
[NOTEBOOK] Run All completed. The interactive video UI above is ready.
